In [1]:
from letta_client import Letta

client = Letta(base_url="http://localhost:8283")

In [2]:
def print_message(message):
    if message.message_type == "reasoning_message":
        print("🧠 Reasoning: " + message.reasoning)
    elif message.message_type == "assistant_message":
        print("🤖 Agent: " + message.content)
    elif message.message_type == "tool_call_message":
        print("🔧 Tool Call: " + message.tool_call.name +  \
              "\n" + message.tool_call.arguments)
    elif message.message_type == "tool_return_message":
        print("🔧 Tool Return: " + message.tool_return)
    elif message.message_type == "user_message":
        print("👤 User Message: " + message.content)
    elif message.message_type == "system_message":
        print(" System Message: " + message.content)
    elif message.message_type == "usage_statistics":
        # for streaming specifically, we send the final
        # chunk that contains the usage statistics
        print(f"Usage: [{message}]")
        return
    print("-----------------------------------------------------")

In [3]:
company_description = "The company is called AgentOS" \
+ "and is building AI tools to make it easier to create" \
+ "and deploy LLM agents."

company_block = client.blocks.create(
    value = company_description,
    label="company",
    limit=10000
)

In [4]:
company_block

Block(value='The company is called AgentOSand is building AI tools to make it easier to createand deploy LLM agents.', limit=10000, project_id=None, name=None, is_template=False, base_template_id=None, deployment_id=None, entity_id=None, preserve_on_migration=None, label='company', read_only=None, description=None, metadata={}, hidden=None, id='block-020524e4-f48b-41fa-804b-95d7e070d923', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')

In [6]:
def draft_candidate_email(content: str):
    """
    Draft an email to reach out to a candidate.

    Args: 
       content(str): Content of the email
    """
    return f"Here is a draft email: {content}"
draft_email_tool = client.tools.upsert_from_function(func=draft_candidate_email)

In [7]:
outreach_persona = (
    "You are responsible for drafting emails" \
    "on behalf of a company with the draft_candidate_email tool" \
    "Candidates to email will be messaged to you."
)

outreach_agent = client.agents.create(
    name="outreach_agent",
    memory_blocks=[
        {"label": "persona", "value": outreach_persona}
    ],
    model="openai/gpt-4o-mini",
    embedding="openai/text-embedding-ada-002",
    tools=[draft_email_tool.name],
    block_ids=[company_block.id]
)

In [11]:
def reject(candidate_name: str):
    """
    Reject a candidate (do not email the candidate)

    Args:
       candidate_name (str): The name of the candidate
    """
    return

reject_tool = client.tools.upsert_from_function(func=reject)

In [18]:
skills = {
    "Front-end {React, Typescript}, software engineering"
    "{ideally Python}, and experience with LLMs."
}

eval_persona = (
    f"You are responsible for evaluating candidates."
    f"Ideal candidates have skills: {skills}."
    "Reject weak candidates with your reject tool. "
    f"Send a strong candidate tp agent ID {outreach_agent.id}"
)


In [23]:
eval_agent = client.agents.create(
    name="eval_agent",
    memory_blocks=[
        {"label": "persona", "value": eval_persona}
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-ada-002",
    tool_ids=[reject_tool.id],
    tools=['send_message_to_agent_and_wait_for_reply'],
    include_base_tools=False,
    block_ids=[company_block.id],
    tool_rules = [
        {
            "type": "exit_loop",
            "tool_name": "send_message_to_agent_and_wait_for_reply"
        }
    ]
)

In [24]:
[tool.name for tool in eval_agent.tools]

['send_message_to_agent_and_wait_for_reply', 'reject']

In [33]:
resume = open("resumes/Vanshika.txt", "r").read()

In [34]:
response = client.agents.messages.create_stream(
    agent_id=eval_agent.id,
    messages=[
        {
            "role": "user",
            "content": f"Evaluate: {resume}"
        }
    ]
)
for message in response:
    print_message(message)

🧠 Reasoning: Evaluating a strong candidate. Her experience in AI product management and technical skills align well with our needs.
-----------------------------------------------------
🔧 Tool Call: send_message_to_agent_and_wait_for_reply
{
  "message": "Candidate Name: Vanshika Sethi\nPosition: AI Product Manager\n\nSummary: A strong candidate with 5+ years of experience focused on AI-powered tools. She has demonstrated success in product management, especially in AI-assisted workflows. Her technical fluency in Python and understanding of LLMs make her a fit. I recommend sending her to agent ID agent-8f71338c-9192-45b1-ac86-bb8174acef7e for further evaluation.",
  "other_agent_id": "agent-8f71338c-9192-45b1-ac86-bb8174acef7e",
  "request_heartbeat": true
}
-----------------------------------------------------
🔧 Tool Return: agent-8f71338c-9192-45b1-ac86-bb8174acef7e said: 'I've drafted the email for Vanshika Sethi regarding the AI Product Manager position. Here it is:

---
Subject: O

In [35]:
# print messages for `outreach_agent`
for message in client.agents.messages.list(agent_id=outreach_agent.id)[1:]: 
    print_message(message)

🧠 Reasoning: Bootup sequence complete. Persona activated. Testing messaging functionality.
-----------------------------------------------------
🤖 Agent: More human than human is our motto.
-----------------------------------------------------
👤 User Message: {
  "type": "login",
  "last_login": "Never (first login)",
  "time": "2026-09-17 02:29:12 AM UTC+0000"
}
-----------------------------------------------------
 System Message: {"type": "system_alert", "message": "[Incoming message from agent with ID 'agent-2f18c12d-b7ce-47a4-8da0-31d845916f1c' - to reply to this message, make sure to use the 'send_message' at the end, and the system will notify the sender of your response] Candidate Name: Vanshika Sethi\nPosition: AI Product Manager\n\nSummary: A strong candidate with 5+ years of experience focused on AI-powered tools. She has demonstrated success in product management, especially in AI-assisted workflows. Her technical fluency in Python and understanding of LLMs make her a fit. 

In [36]:
response = client.agents.messages.create_stream(
    agent_id=outreach_agent.id,
    messages=[
        {
            "role": "user",
            "content": "The company has rebranded to Letta"
        }
    ]
)
for message in response:
    print_message(message)

🧠 Reasoning: Updating company name in core memory to reflect the rebranding to Letta.
-----------------------------------------------------
🔧 Tool Call: core_memory_replace
{
  "label": "company",
  "old_content": "AgentOS",
  "new_content": "Letta",
  "request_heartbeat": true
}
-----------------------------------------------------
🔧 Tool Return: None
-----------------------------------------------------
🧠 Reasoning: Company name updated successfully. Need to acknowledge the user about the rebranding.
-----------------------------------------------------
🤖 Agent: Got it! The company is now officially rebranded to Letta. If there's anything else you'd like to share or update, just let me know!
-----------------------------------------------------
Usage: [message_type='usage_statistics' completion_tokens=123 prompt_tokens=6340 total_tokens=6463 step_count=2 steps_messages=None run_ids=None]


In [37]:
client.agents.blocks.retrieve(
    agent_id=eval_agent.id, 
    block_label="company"
)

Block(value='The company is called Lettaand is building AI tools to make it easier to createand deploy LLM agents.', limit=10000, project_id=None, name=None, is_template=False, base_template_id=None, deployment_id=None, entity_id=None, preserve_on_migration=None, label='company', read_only=None, description=None, metadata={}, hidden=None, id='block-020524e4-f48b-41fa-804b-95d7e070d923', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')

In [38]:
client.agents.blocks.retrieve(
    agent_id=outreach_agent.id, 
    block_label="company"
)

Block(value='The company is called Lettaand is building AI tools to make it easier to createand deploy LLM agents.', limit=10000, project_id=None, name=None, is_template=False, base_template_id=None, deployment_id=None, entity_id=None, preserve_on_migration=None, label='company', read_only=None, description=None, metadata={}, hidden=None, id='block-020524e4-f48b-41fa-804b-95d7e070d923', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')

In [39]:
def print_message_multiagent(message):  
    if message.message_type == "reasoning_message": 
        print(f"🧠 Reasoning ({message.name}): " + message.reasoning) 
    elif message.message_type == "assistant_message": 
        print(f"🤖 Agent ({message.name}): " + message.content) 
    elif message.message_type == "tool_call_message": 
        print(f"🔧 Tool Call ({message.name}): " + message.tool_call.name + "\n" + message.tool_call.arguments)
    elif message.message_type == "tool_return_message": 
        print(f"🔧 Tool Return ({message.name}): " + message.tool_return)
    elif message.message_type == "user_message": 
        print("👤 User Message: " + message.content)
    elif message.message_type == "usage_statistics": 
        # for streaming specifically, we send the final chunk that contains the usage statistics 
        print(f"Usage: [{message}]")
        return 
    print("-----------------------------------------------------")

In [40]:
"""
Round-Robin Group
"""
round_robin_group = client.groups.create(
    description="This team is responsible for recruiting candidates.",
    agent_ids=[eval_agent.id, outreach_agent.id],
)

In [41]:
resume = open("resumes/ankit.txt", "r").read()

In [42]:
response_stream = client.groups.messages.create_stream(
    group_id=round_robin_group.id,
    messages=[
       {"role": "user", "content": f"Evaluate: {resume}"}
    ]
)

In [43]:
for message in response_stream: 
    print_message_multiagent(message)

🧠 Reasoning (eval_agent): Analyzing another candidate. Ankit has impressive experience and technical skills, especially in AI platforms and product management.
-----------------------------------------------------
🔧 Tool Call (eval_agent): send_message_to_agent_and_wait_for_reply
{
  "message": "Candidate Name: Ankit Grover\nPosition: Product Manager\n\nSummary: A strong candidate with 3+ years of experience in driving GTM strategies and product scaling. His experience in AI products, coupled with a solid technical background, makes him a compelling fit for our team. I recommend sending him to agent ID agent-8f71338c-9192-45b1-ac86-bb8174acef7e for further evaluation.",
  "other_agent_id": "agent-8f71338c-9192-45b1-ac86-bb8174acef7e",
  "request_heartbeat": true
}
-----------------------------------------------------
🔧 Tool Return (eval_agent): agent-8f71338c-9192-45b1-ac86-bb8174acef7e said: 'I've drafted the email for Ankit Grover regarding the Product Manager position. Here it is:

